# Circuit Tracer Analysis Backend Demo

This notebook demonstrates the **Interpretune analysis ops pipeline** for circuit-tracer analysis,
running the full **semantic concept intervention** flow as a single composite pipeline:

- **Concept Direction** — Compute a directional concept vector (e.g. "Capitals − States")
- **Attribution Graph** — Generate a circuit-level attribution graph from the concept direction
- **Node Influence** — Score graph nodes by their causal influence on the concept
- **Top Features** — Extract the most influential transcoder features
- **Feature Intervention** — Amplify those features and measure how predictions shift

The five ops above are composed into the `intervention_from_concept` pipeline, which chains
them automatically and returns results in a single `AnalysisBatch`.

> **Tip:** For individual op dispatching and the `DISPATCHER` API, see the
> [op_collection_example](../example_op_collections/op_collection_example.ipynb) notebook.

All analysis uses the **NNsight** backend for circuit-tracer by default.

In [1]:
# @title Imports { display-mode: "form" }
from pprint import pformat

import interpretune as it
from it_examples import _ACTIVE_PATCHES  # noqa: F401
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY
from it_examples.utils.example_helpers import required_os_env
from it_examples.utils.nb_ui_utils import (
    display_token_probs,
    display_top_features_comparison,
    display_topk_token_predictions,
)
from interpretune import ITSession, ITSessionConfig
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.config import AnalysisCfg, init_analysis_cfgs

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
backend = "nnsight"  # Options: "transformerlens", "nnsight"
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
intervention_scale_factor = 10.0  # Multiplier for feature intervention amplitudes

In [3]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

## Session Configuration

We load a registered example module configuration for Gemma-2-2b with the RTE task,
then configure it for the selected backend. The `"gemma"` transcoder set uses Gemma Scope
transcoders, matching the upstream
[attribution_targets_demo](https://github.com/safety-research/circuit-tracer/blob/main/demos/attribution_targets_demo.ipynb).

In [4]:
# Load the demo configuration from the example module registry
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = MODULE_EXAMPLE_REGISTRY.get("gemma2.rte_demo.circuit_tracer")

# Configure backend
base_it_cfg.circuit_tracer_cfg.backend = backend
print(f"Backend: {backend}")

# Optionally override core_log_dir
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir

# Use Gemma Scope transcoders (matches upstream attribution_targets_demo)
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"

# Configure intervention settings
base_it_cfg.circuit_tracer_cfg.intervention_value_source = "top_feature_activation_values"
base_it_cfg.circuit_tracer_cfg.intervention_scale_factor = intervention_scale_factor

print(pformat(base_it_cfg.circuit_tracer_cfg))

Backend: nnsight
CircuitTracerConfig(backend='nnsight',
                    model_name=None,
                    transcoder_set='gemma',
                    dtype=torch.bfloat16,
                    max_n_logits=10,
                    desired_logit_prob=0.95,
                    batch_size=256,
                    max_feature_nodes=8192,
                    offload='cpu',
                    verbose=True,
                    default_node_threshold=0.8,
                    default_edge_threshold=0.98,
                    save_graphs=True,
                    graph_output_dir=None,
                    analysis_target_tokens=['▁Dallas', '▁Austin'],
                    target_token_ids=None,
                    use_neuronpedia=False,
                    intervention_scale_factor=10.0,
                    intervention_value=None,
                    intervention_value_source='top_feature_activation_values',
                    intervention_constrained_layers=None,
                    inter

/home/speediedan/repos/interpretune/src/interpretune/config/transformer_lens.py:261: Interpretune manages the HF model instantiation via `model_name_or_path`. Since `tokenizer_name was not provided, the value provided for `tl_cfg.cfg.tokenizer_name` will be used for `tokenizer_name`.
/home/speediedan/repos/interpretune/src/interpretune/config/transformer_lens.py:261: Interpretune manages the HF model instantiation via `model_name_or_path`. Since `tokenizer_name` was provided, `tl_cfg.cfg.tokenizer_name` will be ignored.
/home/speediedan/repos/interpretune/src/interpretune/config/sae_lens.py:114: use_bridge=True is only meaningful when backend='transformerlens'. This setting will be ignored.
/home/speediedan/repos/interpretune/src/interpretune/config/shared.py:221: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'circuit_tracer_cfg': CircuitTracerConfig(backend='transformerlens', model_name=None, transcoder

In [5]:
# Configure the session with the appropriate adapter composition
if backend == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    adapter_ctx = (it.Adapter.core, it.Adapter.circuit_tracer)

session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

it_session = ITSession(session_cfg)

# Initialize session (loads model, sets up hooks, etc.)
it.it_init(**it_session)

# Set up analysis config on module
module = it_session.module
tokenizer = module.replacement_model.tokenizer

graph_op = it.compute_attribution_graph
module.analysis_cfg = AnalysisCfg(target_op=graph_op, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

print(f"Module type: {type(module).__name__}")
print("Session initialized successfully!")

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight
INFO:interpretune.utils.logging:Loading ReplacementModel with backend: nnsight


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer
INFO:interpretune.utils.logging:NNsight ReplacementModel initialized for Circuit Tracer
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'
INFO:interpretune.utils.logging:Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute '_pre_quantization_dtype'
INFO:interpretune.utils.logging:Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute '_pre_quantization_dtype'
[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule
INFO:interpretune.

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/2490 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/277 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule
INFO:interpretune.utils.logging:Setting up datamodule: InterpretunableDataModule
[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule
INFO:interpretune.utils.logging:Setting up model: InterpretunableModule
[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule
INFO:interpretune.utils.logging:initializing optimizers and schedulers: InterpretunableModule
[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.
INFO:interpretune.utils.logging:Input gradient requirements handled by circuit tracer internally.


Module type: InterpretunableModule
Session initialized successfully!


## Analysis Pipeline

We use the `intervention_from_concept` composite op to chain all five analysis operations
in a single pipeline call. Virtual logit target IDs (from concept-direction attribution
targets) are automatically resolved to real vocabulary token IDs.

| Step | Op | Purpose |
|------|----|---------|
| Pipeline | `concept_direction` | Compute a directional concept vector via paired rejection |
| Pipeline | `compute_attribution_graph` | Generate a circuit-level attribution graph for the prompt |
| Pipeline | `graph_node_influence` | Score graph nodes by causal influence on the concept |
| Pipeline | `extract_top_features` | Rank and extract the most influential transcoder features |
| Pipeline | `feature_intervention_forward` | Amplify top features and measure prediction shifts |

We define the concept groups, prompt, and key tokens, then run the pipeline.

In [6]:
# Concept groups (SentencePiece tokens with leading ▁)
capitals = ["▁Austin", "▁Sacramento", "▁Olympia", "▁Atlanta"]
states = ["▁Texas", "▁California", "▁Washington", "▁Georgia"]
concept_label = "Concept: Capitals − States"

# Analysis prompt
prompt = "Fact: the capital of the state containing Dallas is"

# Key tokens for result comparison (matching upstream attribution_targets_demo)
austin_id = tokenizer.encode("▁Austin", add_special_tokens=False)[-1]
dallas_id = tokenizer.encode("▁Dallas", add_special_tokens=False)[-1]
key_tokens = [("Austin", austin_id), ("Dallas", dallas_id)]

print(f"Prompt: '{prompt}'")
print(f"Key tokens: Austin={austin_id}, Dallas={dallas_id}")

Prompt: 'Fact: the capital of the state containing Dallas is'
Key tokens: Austin=22605, Dallas=26865


In [7]:
# Run the full intervention_from_concept pipeline (all 5 ops in one call)
full_pipeline = it.intervention_from_concept
print(f"Full pipeline: {' → '.join(op.name for op in full_pipeline.composition)}")

results = full_pipeline(
    module,
    AnalysisBatch(
        concept_group_a=capitals,
        concept_group_b=states,
        concept_label=concept_label,
        concept_direction_mode="paired_rejection",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=10,
    intervention_scale_factor=intervention_scale_factor,
)
print("Pipeline complete!")

Full pipeline: concept_direction → compute_attribution_graph → graph_node_influence → extract_top_features → feature_intervention_forward


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.32s
Found 9179 active features
Phase 1: Running forward pass
Forward pass completed in 0.66s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.1072
Will include 8192 of 9179 feature nodes
Input vectors built in 0.76s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 0.13s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:02<00:00, 3018.73it/s]
Feature attributions completed in 2.72s
Attribution completed in 8.03s


Pipeline complete!


## Results: Concept Direction

The concept direction vector captures the "capital-ness" concept via paired rejection,
projecting out state-specific components from each capital embedding.

In [8]:
print(f"Concept direction shape: {results.concept_direction.shape}")
print(f"Concept label: {results.concept_label}")
print(f"Direction norm: {results.concept_direction.norm():.4f}")

Concept direction shape: torch.Size([2304])
Concept label: Concept: Capitals − States
Direction norm: 1.0000


## Results: Top Features

The top transcoder features ranked by node influence scores.
Each feature is a ``(layer, position, feature_index)`` tuple.

> For interactive node influence visualization, see the
> [attribution_analysis](../../analysis_injection/attribution_analysis.ipynb) notebook.

In [9]:
# Build feature tuples and scores for display
features = [tuple(f.tolist()) for f in results.top_feature_ids]
scores = results.top_feature_scores.tolist()

display_top_features_comparison(
    {"Top Features (by node influence)": features},
    {"Top Features (by node influence)": scores},
    neuronpedia_model="gemma-2-2b",
)

#,Node,Score
1,"(21, 10, 5943)",0.0037
2,"(24, 10, 6394)",0.0020
3,"(23, 10, 12237)",0.0018
4,"(24, 10, 5999)",0.0012
5,"(20, 10, 15589)",0.0011
6,"(19, 10, 2695)",0.0010
7,"(24, 10, 6044)",0.0010
8,"(22, 10, 4999)",0.0010
9,"(18, 10, 6101)",0.0009
10,"(0, 2, 16200)",0.0008


## Results: Feature Intervention

Amplifying the top features by the configured scale factor and measuring how
predictions shift. If the pipeline correctly identifies "capital" features,
amplifying them should increase the probability of **Austin** relative to **Dallas**.

In [10]:
pre_logits = results.pre_intervention_logits.float().cpu()
post_logits = results.post_intervention_logits.float().cpu()

# Rich pre/post comparison with key tokens
display_topk_token_predictions(
    prompt,
    pre_logits,
    post_logits,
    tokenizer,
    k=5,
    key_tokens=key_tokens,
)

# Detailed token probability tables
display_token_probs(pre_logits, [austin_id, dallas_id], ["Austin", "Dallas"], title="Pre-Intervention")
display_token_probs(post_logits, [austin_id, dallas_id], ["Austin", "Dallas"], title="Post-Intervention")

# Logit gap analysis
pre_gap = (pre_logits[austin_id] - pre_logits[dallas_id]).item()
post_gap = (post_logits[austin_id] - post_logits[dallas_id]).item()
print("Austin − Dallas logit gap:")
print(f"  Pre-intervention:  {pre_gap:+.4f}")
print(f"  Post-intervention: {post_gap:+.4f}")
print(f"  Change:            {post_gap - pre_gap:+.4f}")

Token,Probability,Distribution
Austin,0.420,42.0%
not,0.057,5.7%
the,0.057,5.7%
Texas,0.050,5.0%
Fort,0.039,3.9%
Token,Probability,Distribution
Austin,0.645,64.5%
San,0.112,11.2%
Austin,0.028,2.8%
Fort,0.025,2.5%


Token,Probability,Logit
Austin,42.045%,26.1250
Dallas,3.046%,23.5000


Token,Probability,Logit
Austin,64.504%,26.1250
Dallas,0.299%,20.7500


Austin − Dallas logit gap:
  Pre-intervention:  +2.6250
  Post-intervention: +5.3750
  Change:            +2.7500


## Summary

The `intervention_from_concept` pipeline ran all five analysis ops in a single call:

| Op | Result |
|----|---------|
| `concept_direction` | Computed a "Capitals − States" concept vector via paired rejection |
| `compute_attribution_graph` | Generated a circuit-level attribution graph for the prompt |
| `graph_node_influence` | Scored graph nodes by causal influence |
| `extract_top_features` | Ranked and extracted the top-10 most influential features |
| `feature_intervention_forward` | Amplified top features and measured prediction shifts |

Virtual logit target IDs from the attribution graph are automatically resolved to real
vocabulary token IDs, so no manual override is needed.

**Key result:** Amplifying the identified "capital" features shifts the model's prediction
towards the correct answer (Austin), validating that the circuit-tracer pipeline identifies
causally relevant features.

### What's Next

- **RTE Concept-Direction Demo**: See `ct_cross_backend_demo.ipynb` for multi-group concept direction validation and bidirectional intervention on RTE examples
- **Analysis Injection**: See `attribution_analysis.ipynb` for node influence visualization via injection hooks
- **Op Dispatching**: See `op_collection_example.ipynb` for the `DISPATCHER` API and individual op usage
- **AnalysisRunner**: For batch analysis workflows, use `AnalysisRunner` with `AnalysisCfg` objects
- **Basic CT Tutorial**: See `circuit_tracer_adapter_example_basic.ipynb` for the foundational adapter tutorial